[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/17_dropout.ipynb)

# 🟢 Easy: Implement Dropout

Implement **Dropout** regularization from scratch.

### Signature
```python
class MyDropout(nn.Module):
    def __init__(self, p: float = 0.5): ...
    def forward(self, x: Tensor) -> Tensor: ...
```

### Rules
- During **training**: zero each element with probability `p`, scale remaining by `1/(1-p)`
- During **eval**: return input unchanged (identity)
- Do NOT use `nn.Dropout` or `F.dropout`

In [ ]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


In [1]:
import torch
import torch.nn as nn

/usr/local/lib/python3.11/site-packages/torch/_subclasses/functional_tensor.py:307: UserWarning: Failed to initialize NumPy: No module named 'numpy' (Triggered internally at /pytorch/torch/csrc/utils/tensor_numpy.cpp:84.)
  cpu = _conversion_method_template(device=torch.device("cpu"))


In [6]:
# ✏️ YOUR IMPLEMENTATION HERE

class MyDropout(nn.Module):
    def __init__(self, p=0.5):
        super().__init__() # nn.Moduleのコンストラクタを呼ぶ
        self.p = p         # ドロップ率を設定する

    def forward(self, x):
        # Dropoutは学習時のみ作動させたい層。
        # 従って、評価モードの場合、もしくはdrop率0の場合はそのままxを返す
        if not self.training or self.p == 0:
            return x
        mask = (torch.rand_like(x) > self.p).float()
        return x * mask / (1 - self.p)

- `torch.rand_like(x)` : `x`と同じshapeで各要素が[0, 1)の一様乱数のテンソルを作成する
- `self.p` : 各要素を`p`と比較してbooleanに変換する。
- `.float()` : `True`->1.0, `False`->0.0に変換する。
- `x * mask` : マスクが0の位置の要素を0にする（=これがdropout）
- `/ (1 - self.p)` : **inverted dropout** と呼ばれるスケーリング


### inverted dropoutのスケーリングが必要な理由
drop後の出力の期待値を、ドロップしないときと同じに保つため。
`p=0.5`の場合を考える。
- 各要素は確率0.5で生き残る
- 単に `x * mask` だと出力の期待値は元の半分になってしまう
- これだと学習時と推論時で次の層への入力スケールが変わってしまう→不整合が生じる

そこで生き残った要素を `1/(1-p)` 倍 (=`p=0.5`ならば2倍) に増幅させてやる:
$$
E[\text{output}] = (1 - p) \cdot \frac{x}{1-p} + p\cdot \frac{0}{1-p} = x
$$

これによって期待値が`x`に戻り、推論時は何もしないだけで正しいスケールになる。だから上述の`if not self.training: return x` で素通しすることができる。学習時と推論時でスケールが一致している為。
(古典的なdropoutでは推論時に出力を`(1-p)`倍する実装もあるが、現代は学習時にスケールしておく**inverted dropout**が標準らしい)

In [7]:
# 🧪 Debug
d = MyDropout(p=0.5)
d.train()
x = torch.ones(10)
print('Train:', d(x))
d.eval()
print('Eval: ', d(x))

Train: tensor([2., 0., 2., 0., 2., 2., 0., 0., 0., 0.])
Eval:  tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [8]:
# ✅ SUBMIT
from torch_judge import check
check('dropout')


🧪 Testing: Implement Dropout (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Eval mode is identity (0.3ms)
  ✅ [2/4] Training: zeros and scaling (1.0ms)
  ✅ [3/4] Drop rate is approximately p (0.6ms)
  ✅ [4/4] Gradient flow (0.2ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (2.2ms total)
  Progress saved. Run status() to see your dashboard.

